# 4. Data Warehouse Design
## Star Schema for Vietnam Real Estate

## 4.1 Star Schema Overview

The data warehouse uses a **Star Schema** design with:
- **Fact Table**: `fact_property` - Contains measures and foreign keys
- **Dimension Tables**: Location, Legal, Furniture, Direction, Property Type, Date

```
                    ┌─────────────────┐
                    │   dim_location  │
                    │  (location_id)  │
                    │     city        │
                    │    district     │
                    └────────┬────────┘
                             │
                             │ 1:N
                             │
    ┌────────────────────────┼────────────────────────┐
    │                        │                        │
    ▼                        ▼                        ▼
┌───────────┐         ┌─────────────┐          ┌────────────┐
│dim_legal  │         │ fact_property│          │dim_furniture│
│(legal_id) │         │  (property_id)│         │(furniture_id)│
│legal_status│        │    area      │          │furniture_state│
└───────────┘         │   frontage   │          └────────────┘
                      │ access_road  │
                      │   floors     │
                      │  bedrooms    │
                      │  bathrooms   │
                      │    price     │
                      └──────┬───────┘
                             │
    ┌────────────────────────┼────────────────────────┐
    │                        │                        │
    ▼                        ▼                        ▼
┌───────────────┐    ┌────────────────┐    ┌──────────────────┐
│  dim_direction │    │   dim_property  │    │    dim_date      │
│(direction_id) │    │(property_type_id│    │   (date_id)      │
│house_direction│    │property_type_name│   │      year        │
│balcony_direction│   └────────────────┘    │     quarter      │
└───────────────┘                           │      month        │
                                           └──────────────────┘
```

## 4.2 Load Data and Prepare for Warehouse

In [ ]:
import pandas as pd
import numpy as np

# Load cleaned data
df = pd.read_csv('../data_cleaned.csv')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

## 4.3 Create Dimension Tables

In [ ]:
# Dimension: Location
dim_location = df[['Address', 'City', 'District']].drop_duplicates().reset_index(drop=True)
dim_location['location_id'] = range(1, len(dim_location) + 1)
dim_location = dim_location[['location_id', 'Address', 'City', 'District']]

print('=== dim_location ===')
print(f'Rows: {len(dim_location)}')
print(dim_location.head())

In [ ]:
# Dimension: Legal
dim_legal = df[['Legal status']].drop_duplicates().reset_index(drop=True)
dim_legal['legal_id'] = range(1, len(dim_legal) + 1)
dim_legal.columns = ['legal_status', 'legal_id']
dim_legal = dim_legal[['legal_id', 'legal_status']]

print('\n=== dim_legal ===')
print(dim_legal)

In [ ]:
# Dimension: Furniture
dim_furniture = df[['Furniture state']].drop_duplicates().reset_index(drop=True)
dim_furniture['furniture_id'] = range(1, len(dim_furniture) + 1)
dim_furniture.columns = ['furniture_state', 'furniture_id']
dim_furniture = dim_furniture[['furniture_id', 'furniture_state']]

print('\n=== dim_furniture ===')
print(dim_furniture)

In [ ]:
# Dimension: Direction
dim_direction = df[['House direction', 'Balcony direction']].drop_duplicates().reset_index(drop=True)
dim_direction['direction_id'] = range(1, len(dim_direction) + 1)
dim_direction = dim_direction[['direction_id', 'House direction', 'Balcony direction']]

print('\n=== dim_direction ===')
print(f'Rows: {len(dim_direction)}')
print(dim_direction.head(10))

In [ ]:
# Dimension: Property Type (derived from data patterns)
property_types = ['Residential', 'Commercial', 'Industrial', 'Agricultural']
dim_property = pd.DataFrame({
    'property_type_id': range(1, len(property_types) + 1),
    'property_type_name': property_types
})

print('\n=== dim_property ===')
print(dim_property)

In [ ]:
# Dimension: Date (simulated from index)
dim_date = pd.DataFrame({
    'date_id': range(1, 13),
    'year': [2024] * 12,
    'quarter': [1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4],
    'month': range(1, 13)
})

print('\n=== dim_date ===')
print(dim_date)

## 4.4 Create Fact Table

In [ ]:
# Create fact table with foreign keys
fact_property = df.copy()

# Merge with dimension tables
fact_property = fact_property.merge(dim_location[['location_id', 'Address', 'City', 'District']], 
                                    on=['Address', 'City', 'District'], how='left')
fact_property = fact_property.merge(dim_legal, left_on='Legal status', right_on='legal_status', how='left')
fact_property = fact_property.merge(dim_furniture, left_on='Furniture state', right_on='furniture_state', how='left')
fact_property = fact_property.merge(dim_direction, on=['House direction', 'Balcony direction'], how='left')

# Add property_type_id (random assignment for demo)
np.random.seed(42)
fact_property['property_type_id'] = np.random.randint(1, 5, len(fact_property))

# Add date_id (random assignment for demo)
fact_property['date_id'] = np.random.randint(1, 13, len(fact_property))

# Add price_per_sqm
fact_property['price_per_sqm'] = fact_property['Price'] / fact_property['Area']

# Select and reorder columns
fact_property = fact_property[[
    'location_id', 'property_type_id', 'legal_id', 'furniture_id',
    'direction_id', 'date_id', 'Area', 'Frontage', 'Access Road',
    'Floors', 'Bedrooms', 'Bathrooms', 'Price', 'price_per_sqm'
]]

# Rename columns
fact_property.columns = [
    'location_id', 'property_type_id', 'legal_id', 'furniture_id',
    'direction_id', 'date_id', 'area', 'frontage', 'access_road',
    'floors', 'bedrooms', 'bathrooms', 'price', 'price_per_sqm'
]

# Add property_id
fact_property.insert(0, 'property_id', range(1, len(fact_property) + 1))

print('=== fact_property ===')
print(f'Rows: {len(fact_property)}')
print(fact_property.head())

## 4.5 Export Schema and Sample Data

In [ ]:
# Save dimension and fact tables
import os
os.makedirs('../datawarehouse', exist_ok=True)

dim_location.to_csv('../datawarehouse/dim_location.csv', index=False)
dim_legal.to_csv('../datawarehouse/dim_legal.csv', index=False)
dim_furniture.to_csv('../datawarehouse/dim_furniture.csv', index=False)
dim_direction.to_csv('../datawarehouse/dim_direction.csv', index=False)
dim_property.to_csv('../datawarehouse/dim_property.csv', index=False)
dim_date.to_csv('../datawarehouse/dim_date.csv', index=False)
fact_property.to_csv('../datawarehouse/fact_property.csv', index=False)

print('Data Warehouse tables exported to: datawarehouse/')

## 4.6 Iceberg Cube Implementation

Iceberg Cubes are OLAP structures that only materialize aggregate combinations exceeding a specified threshold (e.g., COUNT > 20). This reduces storage while maintaining query performance.

In [ ]:
# Iceberg Cube using GROUP BY CUBE
# This generates all possible combinations of the grouping columns

print('=== ICEBERG CUBE: City x Legal Status x Furniture ===\n')

# Query: Average price by City, Legal Status, Furniture State with Iceberg Condition
iceberg_query = """
SELECT 
    dim_l.city,
    dim_leg.legal_status,
    dim_f.furniture_state,
    COUNT(*) as property_count,
    AVG(fp.price) as avg_price,
    MIN(fp.price) as min_price,
    MAX(fp.price) as max_price
FROM fact_property fp
JOIN dim_location dim_l ON fp.location_id = dim_l.location_id
JOIN dim_legal dim_leg ON fp.legal_id = dim_leg.legal_id
JOIN dim_furniture dim_f ON fp.furniture_id = dim_f.furniture_id
GROUP BY CUBE (dim_l.city, dim_leg.legal_status, dim_f.furniture_state)
HAVING COUNT(*) > 20
ORDER BY avg_price DESC;
"""

# Execute using pandas merge operations
merged = fact_property.merge(dim_location, on='location_id') \
                      .merge(dim_legal, on='legal_id') \
                      .merge(dim_furniture, on='furniture_id')

# Group by cube equivalent (all combinations)
from itertools import combinations

dimensions = ['city', 'legal_status', 'furniture_state']
iceberg_results = []

for r in range(1, len(dimensions) + 1):
    for cols in combinations(dimensions, r):
        grouped = merged.groupby(list(cols)).agg({
            'price': ['count', 'mean', 'min', 'max']
        }).reset_index()
        grouped.columns = list(cols) + ['count', 'avg_price', 'min_price', 'max_price']
        grouped = grouped[grouped['count'] > 20]
        iceberg_results.append(grouped)

# Combine all results
all_iceberg = pd.concat(iceberg_results, ignore_index=True)
all_iceberg = all_iceberg.sort_values('avg_price', ascending=False)

print(f'Total aggregate combinations (COUNT > 20): {len(all_iceberg)}')
print('\nTop 10 combinations by average price:')
print(all_iceberg.head(10).to_string(index=False))

### How Iceberg Cubes Work

1. **CUBE Operation**: Generates all possible combinations of grouping columns
2. **Iceberg Condition**: Only keeps combinations where COUNT exceeds threshold (e.g., > 20)
3. **Benefits**: Reduces storage space significantly while maintaining fast queries
4. **Trade-off**: Some rare combinations may be excluded

## 4.7 Sample Analytical Queries

In [ ]:
# Query 1: Average price by city
print('=== QUERY 1: Average Price by City ===\n')
query1 = fact_property.merge(dim_location, on='location_id').groupby('City').agg({
    'price': ['count', 'mean', 'min', 'max']
}).round(2)
query1.columns = ['count', 'avg_price', 'min_price', 'max_price']
query1 = query1.sort_values('avg_price', ascending=False)
print(query1)

In [ ]:
# Query 2: Average price by legal status
print('\n=== QUERY 2: Average Price by Legal Status ===\n')
query2 = fact_property.merge(dim_legal, on='legal_id').groupby('legal_status').agg({
    'price': ['count', 'mean', 'min', 'max']
}).round(2)
query2.columns = ['count', 'avg_price', 'min_price', 'max_price']
query2 = query2.sort_values('avg_price', ascending=False)
print(query2)

In [ ]:
# Query 3: Price by furniture state
print('\n=== QUERY 3: Average Price by Furniture State ===\n')
query3 = fact_property.merge(dim_furniture, on='furniture_id').groupby('furniture_state').agg({
    'price': ['count', 'mean', 'min', 'max']
}).round(2)
query3.columns = ['count', 'avg_price', 'min_price', 'max_price']
query3 = query3.sort_values('avg_price', ascending=False)
print(query3)

In [ ]:
# Query 4: Multi-dimensional analysis (City x Legal Status)
print('\n=== QUERY 4: Average Price by City x Legal Status ===\n')
merged = fact_property.merge(dim_location, on='location_id').merge(dim_legal, on='legal_id')
query4 = merged.groupby(['City', 'legal_status'])['price'].agg(['count', 'mean']).round(2)
query4.columns = ['count', 'avg_price']
query4 = query4[query4['count'] > 10].sort_values('avg_price', ascending=False).head(15)
print(query4)

## 4.8 Data Warehouse Summary

In [ ]:
print('='*60)
print('DATA WAREHOUSE DESIGN - SUMMARY')
print('='*60)
print(f'\nFact Table: {len(fact_property)} rows')
print(f'\nDimension Tables:')
print(f'  - dim_location: {len(dim_location)} rows')
print(f'  - dim_legal: {len(dim_legal)} rows')
print(f'  - dim_furniture: {len(dim_furniture)} rows')
print(f'  - dim_direction: {len(dim_direction)} rows')
print(f'  - dim_property: {len(dim_property)} rows')
print(f'  - dim_date: {len(dim_date)} rows')
print(f'\nSchema Type: Star Schema')
print(f'\nAnalytical Capabilities:')
print('  - Price analysis by location')
print('  - Price analysis by legal status')
print('  - Price analysis by furniture state')
print('  - Multi-dimensional analysis')
print('  - Iceberg Cube for sparse combinations')
print('='*60)